# 02 — Data Preprocessing

Reprojects, clips, resamples, and aligns all raw datasets to the 30m reference grid.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_config, setup_logger, ensure_dirs
from preprocessing import (reproject_raster, clip_raster_to_boundary,
                            resample_raster, align_raster_to_reference,
                            compute_slope, compute_proximity_raster,
                            prepare_study_area)
import yaml, geopandas as gpd

logger = setup_logger("02_preprocessing")
sa_cfg = load_config(ROOT / "config/study_area_config.yml")["study_area"]

RAW       = ROOT / "data/raw"
INTERIM   = ROOT / "data/interim"
PROCESSED = ROOT / "data/processed"
CRS       = sa_cfg["crs_projected"]
RES       = sa_cfg["raster_resolution"]
LGA_NAMES = [l["name"] for l in sa_cfg["lgas"]]

print(f"Target CRS : {CRS}")
print(f"Resolution : {RES}m")
print(f"LGAs       : {len(LGA_NAMES)}")


## Step 1 — Study Area Boundary

In [ ]:
lga_path = RAW / "boundaries/oyo_lga_boundaries.gpkg"

if not lga_path.exists():
    print("LGA boundary file not found.")
    print(f"Expected: {lga_path}")
    print("Run src/data_acquisition.py or download manually.")
else:
    study_area = prepare_study_area(lga_path, LGA_NAMES, CRS)
    study_area.to_file(INTERIM / "study_area.gpkg", driver="GPKG")
    print(f"Study area: {len(study_area)} polygon(s)")
    print(f"Bounds    : {study_area.total_bounds}")
    study_area.plot(figsize=(6,5), color="#1D9E75", alpha=0.5,
                    edgecolor="black", linewidth=0.5)
    import matplotlib.pyplot as plt
    plt.title("Study Area — 19 Rural LGAs, Oyo State")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(ROOT / "outputs/maps/study_area.png", dpi=150)
    plt.show()


## Step 2 — DEM: Reproject, Clip, Derive Slope

In [ ]:
dem_raw = RAW / "dem"
dem_tiles = list(dem_raw.glob("*.tif")) + list(dem_raw.glob("*.TIF"))

if not dem_tiles:
    print("No DEM tiles found in data/raw/dem/")
    print("Download SRTM 1 Arc-Second tiles from: https://earthexplorer.usgs.gov/")
else:
    print(f"Found {len(dem_tiles)} DEM tile(s): {[t.name for t in dem_tiles]}")

    # If multiple tiles, mosaic first
    dem_input = dem_tiles[0]
    if len(dem_tiles) > 1:
        import rasterio
        from rasterio.merge import merge
        datasets = [rasterio.open(t) for t in dem_tiles]
        mosaic, transform = merge(datasets)
        profile = datasets[0].profile.copy()
        profile.update(transform=transform, height=mosaic.shape[1], width=mosaic.shape[2])
        mosaic_path = INTERIM / "dem_mosaic.tif"
        with rasterio.open(mosaic_path, "w", **profile) as dst:
            dst.write(mosaic)
        [d.close() for d in datasets]
        dem_input = mosaic_path
        print(f"Mosaicked {len(dem_tiles)} tiles -> {mosaic_path.name}")

    dem_reproj = reproject_raster(dem_input, INTERIM / "dem_reproj.tif", CRS)
    study_area = gpd.read_file(INTERIM / "study_area.gpkg")
    dem_clipped = clip_raster_to_boundary(dem_reproj, study_area, INTERIM / "dem_clipped.tif")
    dem_30m = resample_raster(dem_clipped, INTERIM / "dem_30m.tif", RES)
    slope_path = compute_slope(dem_30m, INTERIM / "slope_degrees.tif")
    print(f"DEM 30m saved   : {dem_30m}")
    print(f"Slope saved     : {slope_path}")


## Step 3 — Roads Proximity Raster

In [ ]:
roads_path = RAW / "roads/roads_osm.gpkg"

if not roads_path.exists():
    print("Roads not found. Running OSM download...")
    from data_acquisition import download_roads_osm
    study_area = gpd.read_file(INTERIM / "study_area.gpkg")
    roads_path = download_roads_osm(study_area, output_dir=str(RAW / "roads"))

ref_raster = INTERIM / "dem_30m.tif"
if ref_raster.exists() and roads_path.exists():
    compute_proximity_raster(roads_path, ref_raster, INTERIM / "dist_roads.tif")
    print("Roads proximity raster saved.")
else:
    print("Skipping — ref raster or roads not yet available.")


## Step 4 — Health Facilities Proximity Raster

In [ ]:
# Try GRID3 first, fall back to OSM
grid3_files = list((RAW / "health_facilities").glob("*.csv")) +               list((RAW / "health_facilities").glob("*.shp")) +               list((RAW / "health_facilities").glob("*.gpkg"))

if grid3_files:
    hf_path = grid3_files[0]
    print(f"Using GRID3 health facilities: {hf_path.name}")
    # Load and reproject
    if hf_path.suffix == ".csv":
        import pandas as pd
        df = pd.read_csv(hf_path)
        # Common GRID3 column names
        lat_col = next((c for c in df.columns if "lat" in c.lower()), None)
        lon_col = next((c for c in df.columns if "lon" in c.lower() or "lng" in c.lower()), None)
        if lat_col and lon_col:
            hf_gdf = gpd.GeoDataFrame(df,
                geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
                crs="EPSG:4326")
            hf_gdf.to_file(INTERIM / "health_facilities.gpkg", driver="GPKG")
            hf_path = INTERIM / "health_facilities.gpkg"
else:
    print("GRID3 not found. Downloading from OSM...")
    from data_acquisition import download_health_facilities_osm
    study_area = gpd.read_file(INTERIM / "study_area.gpkg")
    hf_path = download_health_facilities_osm(study_area, output_dir=str(RAW / "health_facilities"))

ref_raster = INTERIM / "dem_30m.tif"
if ref_raster.exists() and Path(hf_path).exists():
    compute_proximity_raster(hf_path, ref_raster, INTERIM / "dist_health_facility.tif")
    print("Health facility proximity raster saved.")


## Step 5 — Population, Land Cover, NDVI, Water, Settlements

Repeat reproject → clip → resample for each remaining dataset.

In [ ]:
import rasterio
import numpy as np

def preprocess_raster(name, raw_glob, interp="bilinear"):
    """Generic reproject-clip-resample pipeline for a single raster."""
    from rasterio.enums import Resampling
    raw_dir = RAW / name
    files = list(raw_dir.glob("*.tif")) + list(raw_dir.glob("*.TIF"))
    if not files:
        print(f"  [SKIP] {name} — no files in {raw_dir}")
        return None
    src_path = files[0]
    resamp = Resampling.bilinear if interp == "bilinear" else Resampling.nearest
    reproj = reproject_raster(src_path, INTERIM / f"{name}_reproj.tif", CRS, resamp)
    study_area = gpd.read_file(INTERIM / "study_area.gpkg")
    clipped = clip_raster_to_boundary(reproj, study_area, INTERIM / f"{name}_clipped.tif")
    resampled = resample_raster(clipped, INTERIM / f"{name}_30m.tif", RES, resamp)
    print(f"  [OK ] {name} -> {resampled.name}")
    return resampled

print("Processing continuous rasters (bilinear resampling):")
preprocess_raster("population", RAW.glob("population"))
preprocess_raster("ndvi",       RAW.glob("ndvi"))

print("\nProcessing categorical rasters (nearest-neighbour resampling):")
preprocess_raster("land_cover", RAW.glob("land_cover"), interp="nearest")

print("\nWater bodies proximity:")
water_path = RAW / "water/water_bodies_osm.gpkg"
if not water_path.exists():
    from data_acquisition import download_water_bodies_osm
    study_area = gpd.read_file(INTERIM / "study_area.gpkg")
    water_path = download_water_bodies_osm(study_area, output_dir=str(RAW / "water"))
ref_raster = INTERIM / "dem_30m.tif"
if ref_raster.exists() and water_path.exists():
    compute_proximity_raster(water_path, ref_raster, INTERIM / "dist_water.tif")
    print("  [OK ] water proximity saved")

print("\nSettlements proximity:")
settle_path = RAW / "settlements/settlements_osm.gpkg"
if settle_path.exists() and ref_raster.exists():
    compute_proximity_raster(settle_path, ref_raster, INTERIM / "dist_settlements.tif")
    print("  [OK ] settlements proximity saved")


## Step 6 — Align All Rasters to Reference Grid

In [ ]:
ref = INTERIM / "dem_30m.tif"
to_align = [
    "population_30m.tif",
    "ndvi_30m.tif",
    "land_cover_30m.tif",
    "dist_roads.tif",
    "dist_health_facility.tif",
    "dist_water.tif",
    "dist_settlements.tif",
    "slope_degrees.tif",
]

print("Aligning rasters to reference grid...")
for fname in to_align:
    src = INTERIM / fname
    dst = INTERIM / fname.replace(".tif", "_aligned.tif")
    if src.exists() and ref.exists():
        align_raster_to_reference(src, ref, dst)
    else:
        print(f"  [SKIP] {fname} not yet available")
print("\nPreprocessing complete. All aligned rasters saved to data/interim/")
